In [12]:
# test_subtract_mu.py

import unittest
import numpy as np
import scipy.sparse as sp

import sys
import os

# Define the path to the 'mu_subtract' directory
module_dir = '../../VBPCA/subtract_mu/'

# Check if the path exists
if not os.path.isdir(module_dir):
    raise FileNotFoundError(f"The directory {module_dir} does not exist.")

# Add the directory to sys.path if it's not already included
if module_dir not in sys.path:
    sys.path.append(module_dir)

# Now you can import SUBTRACT_MU directly
from subtract_mu_from_sparse import SUBTRACT_MU


def subtract_mu(Mu, X, M, Xprobe=None, Mprobe=None, update_bias=True):
    """
    Subtracts the bias vector Mu from the data matrix X. If X is sparse,
    uses the SUBTRACT_MU function from the compiled module. Otherwise,
    performs element-wise subtraction.

    Parameters:
    - Mu: (n1,) bias vector.
    - X: (n1, n2) data matrix (sparse or dense).
    - M: (n1, n2) mask matrix.
    - Xprobe: (n1, n2_probe) probe matrix (optional).
    - Mprobe: (n1, n2_probe) probe mask matrix (optional).
    - update_bias: bool flag to determine whether to update X and Xprobe.

    Returns:
    - X_new: Updated data matrix after subtraction.
    - Xprobe_new: Updated probe matrix after subtraction (if provided).
    """
    n2 = X.shape[1]

    if not update_bias:
        return X, Xprobe

    if sp.isspmatrix(X):
        X = subtract_mu_from_sparse(X, Mu)
        if Xprobe is not None and Xprobe.size > 0:
            Xprobe = subtract_mu_from_sparse(Xprobe, Mu)
    else:
        X = X - (Mu[:, np.newaxis] * M)
        if Xprobe is not None and Xprobe.size > 0 and Mprobe is not None:
            Xprobe = Xprobe - (Mu[:, np.newaxis] * Mprobe)

    return X, Xprobe

class TestSubtractMu(unittest.TestCase):
    def test_sparse_update_true(self):
        """
        Test Case 1: Sparse Matrix with update_bias = true
        Description: X is a sparse matrix, and update_bias is true.
        """
        print('Test Case 1: Sparse Matrix with update_bias = true')
        print('-------------------------------------------------')

        # Define Mu vector
        Mu = np.array([1, 2], dtype=np.double)

        # Create a simple sparse matrix X
        X = sp.csr_matrix([[1, 2], [3, 4]], dtype=np.double)

        # Define M matrix (spones(X))
        M = sp.csr_matrix(X.sign(), dtype=np.double)

        # Define Xprobe and Mprobe as empty (not used in this test)
        Xprobe = None
        Mprobe = None

        # Define update_bias
        update_bias = True

        # Expected output
        # Row 1: [1, 2] - 1 = [0, 1]
        # Row 2: [3, 4] - 2 = [1, 2]
        # Replace exact zeros with EPS
        X_expected = sp.csr_matrix([[0, 1], [1, 2]], dtype=np.double)
        EPS = 1e-15
        X_expected_data = X_expected.data.copy()
        X_expected_data[X_expected_data == 0] = EPS
        X_expected = sp.csr_matrix((X_expected_data, X_expected.indices, X_expected.indptr), shape=X_expected.shape)

        # Call the subtract_mu function
        X_new, _ = subtract_mu(Mu, X, M, Xprobe, Mprobe, update_bias)

        # Display results
        print('Original X:')
        print(X.toarray())

        print('\nMu:')
        print(Mu)

        print('\nExpected X_new (with EPS where applicable):')
        print(X_expected.toarray())

        print('\nComputed X_new:')
        print(X_new.toarray())

        # Verify correctness
        self.assertTrue(np.allclose(X_new.toarray(), X_expected.toarray(), atol=1e-12), 'Test Case 1 Failed: Output does not match expected result.')
        print('Test Case 1 Passed: Sparse Matrix with update_bias = true\n')

    def test_sparse_update_false(self):
        """
        Test Case 2: Sparse Matrix with update_bias = false
        Description: X is a sparse matrix, and update_bias is false.
        """
        print('Test Case 2: Sparse Matrix with update_bias = false')
        print('--------------------------------------------------')

        # Define Mu vector
        Mu = np.array([1, 2], dtype=np.double)

        # Create a simple sparse matrix X
        X = sp.csr_matrix([[1, 2], [3, 4]], dtype=np.double)

        # Define M matrix (spones(X))
        M = sp.csr_matrix(X.sign(), dtype=np.double)

        # Define Xprobe and Mprobe as empty (not used in this test)
        Xprobe = None
        Mprobe = None

        # Define update_bias
        update_bias = False

        # Expected output: X should remain unchanged
        X_expected = X.copy()

        # Call the subtract_mu function
        X_new, _ = subtract_mu(Mu, X, M, Xprobe, Mprobe, update_bias)

        # Display results
        print('Original X:')
        print(X.toarray())

        print('\nMu:')
        print(Mu)

        print('\nExpected X_new (unchanged):')
        print(X_expected.toarray())

        print('\nComputed X_new:')
        print(X_new.toarray())

        # Verify correctness
        self.assertTrue(np.allclose(X_new.toarray(), X_expected.toarray(), atol=1e-12), 'Test Case 2 Failed: Output does not match expected result.')
        print('Test Case 2 Passed: Sparse Matrix with update_bias = false\n')

    def test_full_update_true(self):
        """
        Test Case 3: Full (non-sparse) Matrix with update_bias = true
        Description: X is a full matrix, and update_bias is true.
        """
        print('Test Case 3: Full (non-sparse) Matrix with update_bias = true')
        print('-----------------------------------------------------------')

        # Define Mu vector
        Mu = np.array([1, 2], dtype=np.double)

        # Create a simple full matrix X
        X = np.array([[1, 2], [3, 4]], dtype=np.double)

        # Define M matrix (spones(X))
        M = np.sign(X).astype(np.double)

        # Create Xprobe and Mprobe matrices (not used in this test)
        Xprobe = None
        Mprobe = None

        # Define update_bias
        update_bias = True

        # Expected output
        X_expected = X - (Mu[:, np.newaxis] * M)

        # Replace exact zeros with EPS
        EPS = 1e-15
        X_expected[X_expected == 0] = EPS

        # Call the subtract_mu function
        X_new, _ = subtract_mu(Mu, X, M, Xprobe, Mprobe, update_bias)

        # Display results
        print('Original X:')
        print(X)

        print('\nMu:')
        print(Mu)

        print('\nExpected X_new (with EPS where applicable):')
        print(X_expected)

        print('\nComputed X_new:')
        print(X_new)

        # Verify correctness within a tolerance
        self.assertTrue(np.allclose(X_new, X_expected, atol=1e-12), 'Test Case 3 Failed: Output does not match expected result.')
        print('Test Case 3 Passed: Full Matrix with update_bias = true\n')

    def test_full_update_false(self):
        """
        Test Case 4: Full (non-sparse) Matrix with update_bias = false
        Description: X is a full matrix, and update_bias is false.
        """
        print('Test Case 4: Full (non-sparse) Matrix with update_bias = false')
        print('------------------------------------------------------------')

        # Define Mu vector
        Mu = np.array([1, 2], dtype=np.double)

        # Create a simple full matrix X
        X = np.array([[1, 2], [3, 4]], dtype=np.double)

        # Define M matrix (spones(X))
        M = np.sign(X).astype(np.double)

        # Create Xprobe and Mprobe matrices (not used in this test)
        Xprobe = None
        Mprobe = None

        # Define update_bias
        update_bias = False

        # Expected output: X should remain unchanged
        X_expected = X.copy()

        # Call the subtract_mu function
        X_new, _ = subtract_mu(Mu, X, M, Xprobe, Mprobe, update_bias)

        # Display results
        print('Original X:')
        print(X)

        print('\nMu:')
        print(Mu)

        print('\nExpected X_new (unchanged):')
        print(X_expected)

        print('\nComputed X_new:')
        print(X_new)

        # Verify correctness within a tolerance
        self.assertTrue(np.allclose(X_new, X_expected, atol=1e-12), 'Test Case 4 Failed: Output does not match expected result.')
        print('Test Case 4 Passed: Full Matrix with update_bias = false\n')

    def test_all_mu_values_zero(self):
        """
        Test Case 5: All Mu Values Result in Zero After Subtraction
        Description: Subtracting Mu from X results in zeros, which should be replaced by EPS.
        """
        print('Test Case 5: All Mu Values Result in Zero After Subtraction')
        print('------------------------------------------------------------')

        # Define Mu vector
        Mu = np.array([1, 2], dtype=np.double)

        # Create a simple sparse matrix X where X - Mu = 0
        # To achieve X - Mu = 0, set X to have elements equal to Mu per row
        # E.g., Row 1: Mu[0] =1, so X[0,0] =1 and X[0,1]=1
        #       Row 2: Mu[1] =2, so X[1,0] =2 and X[1,1]=2
        X = sp.csr_matrix([[1, 1], [2, 2]], dtype=np.double)

        # Define M matrix (spones(X))
        M = sp.csr_matrix(X.sign(), dtype=np.double)

        # Define Xprobe and Mprobe as empty (not used in this test)
        Xprobe = None
        Mprobe = None

        # Define update_bias
        update_bias = True

        # Expected output
        # X_new should be X - Mu[row] where X - Mu = 0, replaced with EPS
        # Since all elements are zero after subtraction, set all data to EPS
        data = np.full_like(X.data, 1e-15)
        X_expected = sp.csr_matrix((data, X.indices, X.indptr), shape=X.shape)

        # Call the subtract_mu function
        X_new, _ = subtract_mu(Mu, X, M, Xprobe, Mprobe, update_bias)

        # Display results
        print('Original X:')
        print(X.toarray())

        print('\nMu:')
        print(Mu)

        print('\nExpected X_new (with EPS where applicable):')
        print(X_expected.toarray())

        print('\nComputed X_new:')
        print(X_new.toarray())

        # Verify correctness
        self.assertTrue(np.allclose(X_new.toarray(), X_expected.toarray(), atol=1e-12), 'Test Case 5 Failed: Output does not match expected result.')
        print('Test Case 5 Passed: All Mu Values Result in Zero After Subtraction\n')

if __name__ == '__main__':
    unittest.main(argv=[''], exit=False)


.....
----------------------------------------------------------------------
Ran 5 tests in 0.012s

OK


Test Case 5: All Mu Values Result in Zero After Subtraction
------------------------------------------------------------
Original X:
[[1. 1.]
 [2. 2.]]

Mu:
[1. 2.]

Expected X_new (with EPS where applicable):
[[1.e-15 1.e-15]
 [1.e-15 1.e-15]]

Computed X_new:
[[1.e-15 1.e-15]
 [1.e-15 1.e-15]]
Test Case 5 Passed: All Mu Values Result in Zero After Subtraction

Test Case 4: Full (non-sparse) Matrix with update_bias = false
------------------------------------------------------------
Original X:
[[1. 2.]
 [3. 4.]]

Mu:
[1. 2.]

Expected X_new (unchanged):
[[1. 2.]
 [3. 4.]]

Computed X_new:
[[1. 2.]
 [3. 4.]]
Test Case 4 Passed: Full Matrix with update_bias = false

Test Case 3: Full (non-sparse) Matrix with update_bias = true
-----------------------------------------------------------
Original X:
[[1. 2.]
 [3. 4.]]

Mu:
[1. 2.]

Expected X_new (with EPS where applicable):
[[1.e-15 1.e+00]
 [1.e+00 2.e+00]]

Computed X_new:
[[0. 1.]
 [1. 2.]]
Test Case 3 Passed: Full Matrix with update